In [ ]:
import os
import numpy as np
import imageio.v3 as imageio
import torch
import torch.nn.functional as F


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def save_u8(path, img01):
    img01 = np.clip(img01, 0.0, 1.0)
    img8 = (img01 * 255.0 + 0.5).astype(np.uint8)
    imageio.imwrite(path, img8)


def crop_valid(a, border):
    if border <= 0:
        return a
    return a[border:-border, border:-border]


def fftshift(x):
    for dim in (-2, -1):
        n = x.size(dim)
        x = torch.roll(x, shifts=(n // 2,), dims=(dim,))
    return x


def ifftshift(x):
    for dim in (-2, -1):
        n = x.size(dim)
        x = torch.roll(x, shifts=(-(n // 2),), dims=(dim,))
    return x


def create_grid_like_numpy(num_pixels_x, dx, num_pixels_y=None, dy=None, device=None):
    if num_pixels_y is None:
        num_pixels_y = num_pixels_x
    if dy is None:
        dy = dx

    width = dx * num_pixels_x
    height = dy * num_pixels_y

    x = torch.linspace(
        -width / 2,
        width / 2,
        num_pixels_x,
        device=device,
        dtype=torch.float32,
    )

    y = torch.linspace(
        -height / 2,
        height / 2,
        num_pixels_y,
        device=device,
        dtype=torch.float32,
    )

    X, Y = torch.meshgrid(x, y, indexing="xy")
    return X, Y


def crop_center_torch(a, crop_h, crop_w):
    H, W = a.shape
    sx = W // 2 - crop_w // 2
    sy = H // 2 - crop_h // 2
    return a[sy:sy + crop_h, sx:sx + crop_w]


def precompute_angular_spectrum_kernel(ny, nx, dx, wavelength, z, device):
    def qidx(n):
        idx = torch.arange(-n / 2, n / 2, device=device, dtype=torch.float32)
        return (wavelength / dx / n * idx) ** 2

    qx = qidx(nx)
    qy = qidx(ny)[:, None]
    q = qy + qx[None, :]

    if dx < wavelength / np.sqrt(2):
        cond = q > 1.0
        q2 = torch.clamp(q, max=1.0)
        H = torch.exp(2j * np.pi / wavelength * z * torch.sqrt(1.0 - q2))
        H[cond] = 0.0
    else:
        H = torch.exp(2j * np.pi / wavelength * z * torch.sqrt(1.0 - q))

    return ifftshift(H).to(torch.complex64)


def normalize01_detached(x):
    return x / (x.detach().abs().max() + 1e-12)


def create_circular_mask_torch(nx, dx, radius, ny=None, dy=None, device=None):
    if ny is None:
        ny = nx
    if dy is None:
        dy = dx

    width = dx * nx
    height = dy * ny

    x = torch.linspace(-width / 2, width / 2, nx, device=device, dtype=torch.float32)
    y = torch.linspace(-height / 2, height / 2, ny, device=device, dtype=torch.float32)

    x2 = x * x
    y2 = y * y
    R2 = y2[:, None] + x2[None, :]
    return (R2 <= (radius * radius)).to(torch.float32)


def compute_complex_coherence_factor_torch(source_intensity):
    g = fftshift(torch.fft.fft2(ifftshift(source_intensity)))
    return normalize01_detached(g)


def compute_schells_propagation_torch(Ic, gamma):
    tmp = fftshift(torch.fft.ifft2(ifftshift(Ic))) * gamma
    out = fftshift(torch.fft.fft2(ifftshift(tmp)))
    return torch.abs(out)


def gaussian_kernel_1d(window_size, sigma, device):
    coords = (
        torch.arange(window_size, device=device, dtype=torch.float32)
        - (window_size - 1) / 2.0
    )
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / (g.sum() + 1e-12)
    return g


def gaussian_window_2d(window_size, sigma, device):
    g1 = gaussian_kernel_1d(window_size, sigma, device)
    w2 = torch.outer(g1, g1)
    w2 = w2 / (w2.sum() + 1e-12)
    return w2[None, None, :, :]


def ssim_torch(img1, img2, window_size=11, sigma=1.5, data_range=1.0):
    x = img1[None, None, :, :]
    y = img2[None, None, :, :]
    w = gaussian_window_2d(window_size, sigma, img1.device)

    mu_x = F.conv2d(x, w, padding=window_size // 2)
    mu_y = F.conv2d(y, w, padding=window_size // 2)

    mu_x2 = mu_x * mu_x
    mu_y2 = mu_y * mu_y
    mu_xy = mu_x * mu_y

    sigma_x2 = F.conv2d(x * x, w, padding=window_size // 2) - mu_x2
    sigma_y2 = F.conv2d(y * y, w, padding=window_size // 2) - mu_y2
    sigma_xy = F.conv2d(x * y, w, padding=window_size // 2) - mu_xy

    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2

    num = (2 * mu_xy + C1) * (2 * sigma_xy + C2)
    den = (mu_x2 + mu_y2 + C1) * (sigma_x2 + sigma_y2 + C2)

    return (num / (den + 1e-12)).mean()


def build_unit_object(a, b, unit_eps):
    denom = torch.sqrt(a * a + b * b + unit_eps)
    return (a / denom).to(torch.complex64) + 1j * (b / denom).to(torch.complex64)


def enforce_phase_0_to_pi(b_tensor):
    with torch.no_grad():
        b_tensor.clamp_(min=0.0)


def obj_to_phase01(obj):
    phi = torch.atan2(obj.imag, obj.real)
    phi = torch.clamp(phi, 0.0, np.pi)
    phase01 = phi / np.pi
    return phase01.clamp(0.0, 1.0).to(torch.float32)


def background_suppression_weight(
    it,
    n_iters,
    lambda_bg_phase,
    ramp_start_frac,
    ramp_end_frac,
):
    if n_iters <= 1:
        return lambda_bg_phase

    t = float(it - 1) / float(n_iters - 1)
    ramp = (t - ramp_start_frac) / max(1e-12, ramp_end_frac - ramp_start_frac)
    ramp = max(0.0, min(1.0, ramp))

    return lambda_bg_phase * ramp


def phase_background_suppression_loss(
    obj,
    bg_phase_gate_frac_pi,
    bg_grad_gate_rad,
):
    phase01 = obj_to_phase01(obj)
    phi = phase01 * np.pi

    dx_phi = F.pad(phi[:, 1:] - phi[:, :-1], (0, 1, 0, 0))
    dy_phi = F.pad(phi[1:, :] - phi[:-1, :], (0, 0, 0, 1))

    with torch.no_grad():
        grad_mag = torch.sqrt(
            dx_phi.detach() * dx_phi.detach()
            + dy_phi.detach() * dy_phi.detach()
            + 1e-12
        )

        phase_gate = torch.exp(
            -(
                phase01.detach()
                / max(1e-12, bg_phase_gate_frac_pi)
            ) ** 2
        )

        grad_gate = torch.exp(
            -(
                grad_mag
                / max(1e-12, bg_grad_gate_rad)
            ) ** 2
        )

        bg_gate = phase_gate * grad_gate

    bg_loss = torch.sum(bg_gate * phase01 * phase01) / (torch.sum(bg_gate) + 1e-12)

    return bg_loss


def gvs_forward_obj_normalized(
    obj_2000,
    gamma,
    H_as,
    pad,
    phase_factor,
    num_px,
):
    field_pad = F.pad(obj_2000, (pad, pad, pad, pad), mode="constant", value=1.0)
    Uc = torch.fft.ifft2(torch.fft.fft2(field_pad * phase_factor) * H_as)
    Ic = (torch.abs(Uc) ** 2).to(torch.float32)
    Igvs_full = compute_schells_propagation_torch(Ic, gamma)
    I_crop = crop_center_torch(Igvs_full, num_px, num_px).to(torch.float32)
    return normalize01_detached(I_crop)


def coherent_forward_obj_normalized(
    obj_2000,
    H_as,
    pad,
    phase_factor,
    num_px,
):
    field_pad = F.pad(obj_2000, (pad, pad, pad, pad), mode="constant", value=1.0)
    Uc = torch.fft.ifft2(torch.fft.fft2(field_pad * phase_factor) * H_as)
    Ic = (torch.abs(Uc) ** 2).to(torch.float32)
    I_crop = crop_center_torch(Ic, num_px, num_px).to(torch.float32)
    return normalize01_detached(I_crop)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

wavelength = ## Define your wavelength here ##
z_source_object = ## Define your source-object distance here ##
dx_camera = ## Define your camera pixel size here ##

num_px = ## Define your number of pixels here ##
pad_factor = ## Define your padding factor here ##
pad = pad_factor * num_px
zpad = num_px + 2 * pad
ny = nx = zpad

source_radii = ## [Define your source radii here] ##

coh_names = ## [Define your coherence names here] ##

meas_root = "REPLACE_WITH_MEASUREMENT_ROOT"
output_root = "REPLACE_WITH_OUTPUT_ROOT"

object_groups = ## [Define your object groups here] ##

object_ids = ## [Define your object IDs here] ##

selected_object_ids_by_group = {obj_group: object_ids for obj_group in object_groups}

z_list_mm = [2, 4]
z_list_m = [z_mm * 1e-3 for z_mm in z_list_mm]

n_iters = ## Define your number of iterations here ##
lr = ## Define your learning rate here ##

crop_border = ## Define your crop border here ##

use_ssim_loss = True
w_mse = 0.10
w_ssim = 1.00

lambda_bg_phase = ## Define your background phase suppression weight here ##
bg_phase_gate_frac_pi = ## Define your background phase gate fraction (in units of pi) here ##
bg_grad_gate_rad = ## Define your background gradient gate (in radians) here ##
bg_ramp_start_frac = ## Define your background ramp start fraction here ##
bg_ramp_end_frac = ## Define your background ramp end fraction here ##

unit_eps = 1e-8

ensure_dir(output_root)

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Precomputing phase factor and angular-spectrum kernels.")

X, Y = create_grid_like_numpy(nx, dx_camera, ny, dx_camera, device=device)

phase_factor = torch.exp(
    1j * np.pi / (wavelength * z_source_object) * (X ** 2 + Y ** 2)
).to(torch.complex64)

H_as_by_zmm = {}
dx_src_by_zmm = {}

for z_mm, z_m in zip(z_list_mm, z_list_m):
    H_as_by_zmm[z_mm] = precompute_angular_spectrum_kernel(
        ny,
        nx,
        dx_camera,
        wavelength,
        z_m,
        device=device,
    )

    dx_src_by_zmm[z_mm] = dx_camera * z_source_object / z_m

for obj_group in object_groups:
    for obj_id in selected_object_ids_by_group[obj_group]:
        print("=" * 100)
        print(f"OBJECT: {obj_group}/{obj_id}")
        print("=" * 100)

        for coh_name, radius in zip(coh_names, source_radii):
            measurements_by_zmm = {}
            missing_measurement = False

            for z_mm in z_list_mm:
                meas_path = os.path.join(
                    meas_root,
                    obj_group,
                    obj_id,
                    coh_name,
                    f"Simulated_{z_mm}mm.png",
                )

                if not os.path.exists(meas_path):
                    print(f"Missing measurement: {meas_path}")
                    missing_measurement = True
                    break

                I_np = imageio.imread(meas_path, mode="L").astype(np.float32) / 255.0
                I_np = (I_np - float(I_np.min())) / max(
                    1e-8,
                    float(I_np.max() - I_np.min()),
                )

                measurements_by_zmm[z_mm] = torch.from_numpy(I_np).to(device)

            if missing_measurement:
                continue

            print("-" * 100)
            print(
                f"MEASUREMENT SET: {obj_group}/{obj_id}/{coh_name} "
                f"(source_radius = {radius * 1e6:.1f} um)"
            )
            print("-" * 100)

            out_dir = os.path.join(output_root, obj_group, obj_id, coh_name)
            ensure_dir(out_dir)

            gamma_by_zmm = {}

            with torch.no_grad():
                for z_mm in z_list_mm:
                    dx_src = dx_src_by_zmm[z_mm]

                    src_I = create_circular_mask_torch(
                        nx,
                        dx_src,
                        radius,
                        ny,
                        dx_src,
                        device=device,
                    )

                    gamma_by_zmm[z_mm] = compute_complex_coherence_factor_torch(
                        src_I
                    ).to(torch.complex64)

            a = torch.ones(
                (num_px, num_px),
                device=device,
                dtype=torch.float32,
                requires_grad=True,
            )

            b = torch.zeros(
                (num_px, num_px),
                device=device,
                dtype=torch.float32,
                requires_grad=True,
            )

            opt = torch.optim.Adam([a, b], lr=lr)

            for it in range(1, n_iters + 1):
                opt.zero_grad()

                obj = build_unit_object(a, b, unit_eps)

                mse_sum = 0.0
                ssim_sum = 0.0

                for z_mm in z_list_mm:
                    I_meas_z = measurements_by_zmm[z_mm]

                    I_pred_z = gvs_forward_obj_normalized(
                        obj,
                        gamma_by_zmm[z_mm],
                        H_as_by_zmm[z_mm],
                        pad,
                        phase_factor,
                        num_px,
                    )

                    mse_z = torch.mean((I_pred_z - I_meas_z) ** 2)
                    mse_sum = mse_sum + mse_z

                    if use_ssim_loss:
                        ssim_sum = ssim_sum + ssim_torch(
                            I_pred_z,
                            I_meas_z,
                            window_size=11,
                            sigma=1.5,
                            data_range=1.0,
                        )

                mse = mse_sum / len(z_list_mm)

                if use_ssim_loss:
                    ssim_val = ssim_sum / len(z_list_mm)
                    data_loss = w_mse * mse + w_ssim * (1.0 - ssim_val)
                else:
                    data_loss = mse

                bg_loss = phase_background_suppression_loss(
                    obj,
                    bg_phase_gate_frac_pi,
                    bg_grad_gate_rad,
                )

                bg_weight = background_suppression_weight(
                    it,
                    n_iters,
                    lambda_bg_phase,
                    bg_ramp_start_frac,
                    bg_ramp_end_frac,
                )

                loss = data_loss + bg_weight * bg_loss

                loss.backward()
                opt.step()

                enforce_phase_0_to_pi(b)

                if it == 1 or it % 20 == 0:
                    print(f"[GVS] Iter {it:04d}/{n_iters:04d} Loss={float(loss.item()):.6e}")

            with torch.no_grad():
                obj_gvs = build_unit_object(a, b, unit_eps)
                phase01_gvs = obj_to_phase01(obj_gvs)

            a2 = torch.ones(
                (num_px, num_px),
                device=device,
                dtype=torch.float32,
                requires_grad=True,
            )

            b2 = torch.zeros(
                (num_px, num_px),
                device=device,
                dtype=torch.float32,
                requires_grad=True,
            )

            opt2 = torch.optim.Adam([a2, b2], lr=lr)

            for it in range(1, n_iters + 1):
                opt2.zero_grad()

                obj_c = build_unit_object(a2, b2, unit_eps)

                mse_sum = 0.0
                ssim_sum = 0.0

                for z_mm in z_list_mm:
                    I_meas_z = measurements_by_zmm[z_mm]

                    I_pred_z = coherent_forward_obj_normalized(
                        obj_c,
                        H_as_by_zmm[z_mm],
                        pad,
                        phase_factor,
                        num_px,
                    )

                    mse_z = torch.mean((I_pred_z - I_meas_z) ** 2)
                    mse_sum = mse_sum + mse_z

                    if use_ssim_loss:
                        ssim_sum = ssim_sum + ssim_torch(
                            I_pred_z,
                            I_meas_z,
                            window_size=11,
                            sigma=1.5,
                            data_range=1.0,
                        )

                mse = mse_sum / len(z_list_mm)

                if use_ssim_loss:
                    ssim_val = ssim_sum / len(z_list_mm)
                    data_loss = w_mse * mse + w_ssim * (1.0 - ssim_val)
                else:
                    data_loss = mse

                bg_loss = phase_background_suppression_loss(
                    obj_c,
                    bg_phase_gate_frac_pi,
                    bg_grad_gate_rad,
                )

                bg_weight = background_suppression_weight(
                    it,
                    n_iters,
                    lambda_bg_phase,
                    bg_ramp_start_frac,
                    bg_ramp_end_frac,
                )

                loss = data_loss + bg_weight * bg_loss

                loss.backward()
                opt2.step()

                enforce_phase_0_to_pi(b2)

                if it == 1 or it % 20 == 0:
                    print(f"[COH] Iter {it:04d}/{n_iters:04d} Loss={float(loss.item()):.6e}")

            with torch.no_grad():
                obj_coh = build_unit_object(a2, b2, unit_eps)
                phase01_coh = obj_to_phase01(obj_coh)

            phase_gvs_crop = crop_valid(
                phase01_gvs,
                crop_border,
            ).detach().cpu().numpy()

            phase_coh_crop = crop_valid(
                phase01_coh,
                crop_border,
            ).detach().cpu().numpy()

            save_u8(os.path.join(out_dir, "phase_GVS.png"), phase_gvs_crop)
            save_u8(os.path.join(out_dir, "phase_COH.png"), phase_coh_crop)

            del gamma_by_zmm
            del a, b, a2, b2
            del obj_gvs, obj_coh
            del phase01_gvs, phase01_coh
            del measurements_by_zmm

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

print("Done. Reconstructions saved under:", os.path.abspath(output_root))
